In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Hello World: Circuit Quantum Pertamamu

Buat sebuah [Bell state](https://en.wikipedia.org/wiki/Bell_state) (dua qubit yang saling terjerat) dan jalankan dengan tiga cara:

1. **Simulasi ideal** — hasil sempurna, tidak perlu akun
2. **Simulasi dengan noise** — mensimulasikan perangkat nyata, tidak perlu akun
3. **Hardware quantum sungguhan** — memerlukan akun IBM Quantum gratis (langkah penyiapan ada di bawah)

## Bangun Circuit

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

## Opsi 1: Simulasi ideal (tidak perlu akun)
Menggunakan `StatevectorSampler` — simulator lokal dengan hasil sempurna tanpa noise.

In [ ]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## Opsi 2: Simulasi dengan noise (tidak perlu akun)
Menggunakan `FakeManilaV2` — simulator lokal yang meniru perangkat quantum IBM sungguhan, termasuk karakteristik noise-nya. Circuit harus ditranspilasi terlebih dahulu (disesuaikan) dengan gate set dan konektivitas qubit perangkat tersebut.

In [ ]:
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeManilaV2()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Opsi 3: Hardware quantum sungguhan
Memerlukan akun IBM Quantum gratis. Untuk membuatnya:

1. Daftar di [quantum.cloud.ibm.com/registration](https://quantum.cloud.ibm.com/registration) — tidak perlu kartu kredit untuk 30 hari pertama
2. Masuk di [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) dan pilih region **us-east** (wajib untuk Open Plan gratis)
3. Buat sebuah instance (Open Plan gratis) di [Instances](https://quantum.cloud.ibm.com/instances) jika kamu belum punya
4. Buat API key di [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) (atau di [cloud.ibm.com/iam/apikeys](https://cloud.ibm.com/iam/apikeys))
5. Salin **CRN** (Cloud Resource Name) kamu dari halaman [Instances](https://quantum.cloud.ibm.com/instances)

Lalu, jika kamu belum menyimpan kredensial di sesi Binder ini, jalankan sel di bawah. Ganti `<your-api-key>` dengan API key dari langkah 4, dan `<your-crn>` dengan CRN dari langkah 5.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="<your-api-key>",
    instance="<your-crn>",
    set_as_default=True,
    overwrite=True,
)

**Catatan:** Job pada hardware sungguhan mungkin membutuhkan waktu tergantung antrean. Jika sel masih berjalan, kamu bisa memeriksa status job dan melihat hasilnya di [quantum.cloud.ibm.com/workloads](https://quantum.cloud.ibm.com/workloads?user=me).

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Langkah selanjutnya?
- **[Tutorials](https://doqumentation.org/tutorials)** — panduan langkah demi langkah tentang algoritma, mitigasi error, transpilasi, dan lainnya
- **[Guides](https://doqumentation.org/guides)** — materi referensi tentang menjalankan circuit, primitives, dan Qiskit Runtime
- **[Courses](https://doqumentation.org/learning/courses/basics-of-quantum-information)** — jalur belajar terstruktur dari dasar quantum hingga komputasi skala utilitas
- **[Modules](https://doqumentation.org/learning/modules/computer-science)** — modul konseptual lebih mendalam tentang ilmu komputer dan mekanika quantum